In [ ]:
import pandas as pd, pyreadstat
df, meta = pyreadstat.read_sav("../lfs_2023_mon.sav", apply_value_formats=True)
df.shape

In [ ]:
labels = pd.Series(meta.column_names_to_labels)
pd.set_option("display.max_rows", 200)
labels

In [ ]:
keys = "цалин|орлого|ажил|салбар|эрхэл|боловсрол|төгрөг|цаг"
labels[labels.str.contains(keys, case=False, na=False)]

In [ ]:
labels.to_csv("variables_2023.csv", encoding="utf-8-sig")

In [ ]:
pd.set_option("display.max_colwidth", None)        # тайлбарыг бүтнээр нь харуулна
labels[labels.index.str.match(r"^f\d")]            # F хэсгийн бүх багана

In [ ]:
f_cols = [c for c in df.columns if c.startswith("f0")]
df[f_cols].dropna(how="all").head(10)

In [ ]:
print(df["f01"].value_counts(dropna=False), "\n")
print(df["f03"].value_counts(dropna=False), "\n")
df["f02"].describe(percentiles=[.01, .25, .5, .75, .99])

In [ ]:
import numpy as np

w = df[df["f03"] == "Нэг сард"].dropna(subset=["f02"]).copy()
w["цалин"] = w["f02"] * 1000                       # төгрөг рүү

def w_median(x, wt):                               # жинлэсэн медиан
    i = np.argsort(x); x, wt = np.asarray(x)[i], np.asarray(wt)[i]
    return x[np.searchsorted(np.cumsum(wt), wt.sum() / 2)]

w.groupby("a03").apply(lambda g: pd.Series({
    "n": len(g),
    "дундаж": np.average(g["цалин"], weights=g["weight_cal_Y"]),
    "медиан": w_median(g["цалин"], g["weight_cal_Y"]),
}))

In [ ]:
import numpy as np

factor = {"Нэг сард": 1, "Хоёр долоо хоногт": 2.17,
          "Долоо хоногт": 4.33, "Нэг өдөр": 21.7, "Улирлын төгсгөлд": 1/3}

w = df[(df["f01"] == "Тийм, мөнгөн хэлбэрээр авсан") & (df["f02"] > 0)].copy()
w["цалин"] = w["f02"] * w["f03"].astype(str).map(factor) * 1000   # сарын, төгрөгөөр
w = w.dropna(subset=["цалин"])                                    # «Бусад» гэх мэтийг хасна

# Хэт утга: цалин хазгай тул лог дээр MAD аргаар шалгана
lw  = np.log(w["цалин"])
mad = (lw - lw.median()).abs().median()
w["хэт_утга"] = (0.6745 * (lw - lw.median()) / mad).abs() > 3.5

print(len(w), "цалинтай ажилтан;", w["хэт_утга"].sum(), "хэт утга")
w["цалин"].describe(percentiles=[.01, .25, .5, .75, .99]).round(0)

In [ ]:
pd.options.display.float_format = "{:,.0f}".format

In [ ]:
from scipy import stats
import seaborn as sns, matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Arial"            # монгол үсэг

эр = np.log(w.loc[w["a03"] == "Эрэгтэй", "цалин"])
эм = np.log(w.loc[w["a03"] == "Эмэгтэй", "цалин"])
print(stats.ttest_ind(эр, эм, equal_var=False))  # Welch t-тест (лог цалин дээр)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=w, x="a03", y="цалин", log_scale=True, showfliers=False, ax=ax)
ax.set(title="2023 оны сарын цалин, хүйсээр", xlabel="", ylabel="Цалин (₮, лог шкал)")
fig.savefig("figures/цалин_хүйс_2023.png", dpi=300, bbox_inches="tight")


In [ ]:
# Асуулт 1. Ажиллах хүчний бүтэц. Эхлээд хөдөлмөрийн статусын баганыг олно:
labels[labels.str.contains("ажиллах хүч|ажилгүй|идэвх|статус", case=False, na=False)]

In [ ]:
df["e04"].value_counts(dropna=False)

In [ ]:
pd.crosstab(df["h01a"], df["h11"], dropna=False)

In [ ]:
ажилгүй = ~ажиллагч & (d["h01a"] == "Тийм") & (d["h11"] != "Үгүй")
d["статус"] = np.select([ажиллагч, ажилгүй],
                        ["Ажиллагч", "Ажилгүй"], default="Ажиллах хүчнээс гадуур")

n = d["статус"].value_counts()
АХ = n["Ажиллагч"] + n["Ажилгүй"]
print(n, "\n")
print(f"Оролцооны түвшин: {АХ / len(d):.1%}")
print(f"Ажилгүйдлийн түвшин: {n['Ажилгүй'] / АХ:.1%}")

In [ ]:
import os
os.makedirs("figures", exist_ok=True)
short = {"Хэн нэгэнд цалин хөлстэй ажиллагч": "Цалинтай ажиллагч",
         "Хувийн бизнесийн үйл ажиллагаа": "Хувиараа эрхлэгч",
         "Өрхийн гишүүдийн болон өрхийн хийж байгаа бизнесийн үйл ажиллага": "Өрхийн бизнес"}
e = d["e04"].map(short).fillna("Бусад").where(ажиллагч).value_counts(normalize=True)
s = d["статус"].value_counts(normalize=True)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, data, title in [(ax[0], s, "15+ насны хүн ам"), (ax[1], e, "Ажиллагчид, байдлаар")]:
    bars = a.barh(data.index[::-1], data.values[::-1] * 100, color="#2a78d6")
    a.bar_label(bars, fmt="%.1f%%", padding=3)
    a.set(title=title, xlabel="%", xlim=(0, 70))
fig.suptitle("Ажиллах хүчний бүтэц, 2023", weight="bold")
fig.tight_layout(); fig.savefig("figures/q1_бүтэц.png", dpi=300, bbox_inches="tight")

In [ ]:

import os, matplotlib.pyplot as plt
os.makedirs("figures", exist_ok=True)
plt.rcParams["font.family"] = "Arial"

s = w["цалин"] / 1e6                                   # сая төгрөгөөр
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(s, bins=np.arange(0, 6.05, 0.1), range=(0, 6), color="#2a78d6", edgecolor="white")
ax[0].axvline(s.median(), color="black", ls="--", label=f"Медиан {s.median():.2f} сая")
ax[0].axvline(s.mean(), color="#eb6834", label=f"Дундаж {s.mean():.2f} сая")
ax[0].set(title="Сарын цалин", xlabel="Сая ₮", ylabel="Хүний тоо"); ax[0].legend()
ax[1].hist(np.log10(w["цалин"]), bins=40, color="#2a78d6", edgecolor="white")
ax[1].set(title="Лог хувиргалтын дараа", xlabel="np.log10(w['цалин'])", ylabel="Хүний тоо")
ax[1].set_xlim(5, 7.5) 
ax[1].set_xticks([5, 5.7, 6, 6.3, 7],
                 ["100 мян", "500 мян", "1 сая", "2 сая", "10 сая"])
fig.tight_layout(); fig.savefig("figures/q2_цалин_тархалт.png", dpi=300, bbox_inches="tight")

In [ ]:
# Асуулт 3. Хүйс, нас, боловсролоор
w["нас"] = pd.to_numeric(w["a05"], errors="coerce")
w["насны_бүлэг"] = pd.cut(w["нас"], [14, 24, 34, 44, 54, 64, 100],
                          labels=["15–24", "25–34", "35–44", "45–54", "55–64", "65+"])

for col in ["a03", "насны_бүлэг", "a28"]:
    display(w.groupby(col, observed=True)["цалин"]
             .agg(n="count", медиан="median", дундаж="mean").round(0))

In [ ]:
order = ["Эрэгтэй", "Эмэгтэй"]
tab = (w.groupby("a03", observed=True)["цалин"]
         .agg(n="count", median="median", mean="mean", std="std")
         .reindex(order))
tab[["median", "mean", "std"]] /= 1e6
gap = 1 - tab.loc["Эмэгтэй", "median"] / tab.loc["Эрэгтэй", "median"]

print(f"Эмэгтэйчүүдийн median цалин {gap:.0%}-иар бага")
tab.T.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
bars = ax.bar(order, tab["median"], width=0.6, color=["#134e5e", "#e36414"])
ax.bar_label(bars, fmt="%.1f", padding=6, fontsize=12)
ax.set_ylim(0, 2.5)                                  # 2025 оны слайдтай ижил муж
ax.set_title("Median сарын цалин, сая ₮", fontsize=12)
ax.yaxis.grid(True, color="#e5e5e5"); ax.set_axisbelow(True)
ax.tick_params(axis="y", labelleft=False, length=0)
ax.tick_params(axis="x", length=0, labelsize=12, labelcolor="#555555")
for sp in ax.spines.values():
    sp.set_visible(False)
fig.tight_layout()
fig.savefig("figures/q3_хүйс_2023.png", dpi=300, bbox_inches="tight", facecolor="white")

In [ ]:
def clean_wages(df):
    factor = {"Нэг сард": 1, "Хоёр долоо хоногт": 2.17,
              "Долоо хоногт": 4.33, "Нэг өдөр": 21.7, "Улирлын төгсгөлд": 1/3}
    w = df[(df["f01"] == "Тийм, мөнгөн хэлбэрээр авсан") & (df["f02"] > 0)].copy()
    w["цалин"] = w["f02"] * w["f03"].astype(str).map(factor) * 1000
    return w.dropna(subset=["цалин"])

df25, meta25 = pyreadstat.read_dta("../lfs_2025_mon.dta", apply_value_formats=True)
print(df25["f02"].describe())          # нэгж мянган ₮ хэвээр байгаа эсэх
w25 = clean_wages(df25)

summary = lambda w: (w.groupby("a03")["цалин"]
                      .agg(n="count", median="median", mean="mean", std="std")
                      / [1, 1e6, 1e6, 1e6]).round(2)
display(summary(clean_wages(df)), summary(w25))   # 2023 ба 2025
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(summary(clean_wages(df)), summary(w25))

In [ ]:
vc = w25["f03"].value_counts()
print(vc, "\n")
print("«Нэг сард»-аас бусад:", vc.sum() - vc.get("Нэг сард", 0))
print("Давхардсан мөр:", df25.duplicated().sum())
w25.loc[w25["f03"] == "Нэг сард", "a03"].value_counts()
m = w25[w25["f03"] == "Нэг сард"].copy()
m["нас"] = pd.to_numeric(m["a05"], errors="coerce")

tests = {
    "65+ нас":                m["нас"] >= 65,
    "15-аас доош нас":        m["нас"] < 15,
    "Цалин < 100 мянга":      m["f02"] < 100,
    "Цалин < 300 мянга":      m["f02"] < 300,
    "Цалин > 10 сая":         m["f02"] > 10000,
    "Цалинтай ажиллагч биш":  m["e04"] != "Хэн нэгэнд цалин хөлстэй ажиллагч",
}
for name, mask in tests.items():
    print(f"{name:25}", m.loc[mask, "a03"].value_counts().reindex(["Эрэгтэй", "Эмэгтэй"]).tolist())

In [ ]:
# 2023: Хүйсээр цалин (2025 оны слайдтай ижил дүрэм)
w23 = clean_wages(df)
w23 = w23[w23["f03"] == "Нэг сард"]                 # зөвхөн сарын цалинтай

order = ["Эрэгтэй", "Эмэгтэй"]
tab = (w23.groupby("a03", observed=True)["цалин"]
          .agg(n="count", median="median", mean="mean", std="std")
          .reindex(order))
tab[["median", "mean", "std"]] /= 1e6
gap = 1 - tab.loc["Эмэгтэй", "median"] / tab.loc["Эрэгтэй", "median"]

print(f"Гарчиг: Эмэгтэйчүүдийн median цалин {gap:.0%}-иар бага")
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(tab.T)                                  # слайдын хүснэгтэд оруулах тоо

# График
fig, ax = plt.subplots(figsize=(6, 5.5))
bars = ax.bar(order, tab["median"], width=0.6, color=["#134e5e", "#e36414"])
ax.bar_label(bars, fmt="%.2f", padding=6, fontsize=12)
ax.set_ylim(0, 2.5)                                 # 2025 оны слайдтай ижил муж
ax.set_title("Median сарын цалин, сая ₮", fontsize=12)
ax.yaxis.grid(True, color="#e5e5e5"); ax.set_axisbelow(True)
ax.tick_params(axis="y", labelleft=False, length=0)
ax.tick_params(axis="x", length=0, labelsize=12, labelcolor="#555555")
for sp in ax.spines.values():
    sp.set_visible(False)
fig.tight_layout()
fig.savefig("figures/q3_хүйс_2023.png", dpi=300, bbox_inches="tight", facecolor="white")

In [ ]:
# 2023: Боловсролоор цалин (2025 оны слайдтай ижил загвар)
order = ["Доктор", "Магистр", "Бакалаврын дээд", "Дипломын дээд",
         "Тусгай мэргэжлийн дунд", "Техникийн болон мэргэжлийн",
         "Бүрэн дунд", "Суурь", "Бага", "Боловсролгүй"]
dark, mid, light = "#134e5e", "#5f8f99", "#a0b8be"
color = {"Магистр": dark, "Бакалаврын дээд": dark,
         "Дипломын дээд": mid, "Тусгай мэргэжлийн дунд": mid,
         "Техникийн болон мэргэжлийн": mid, "Бүрэн дунд": mid}

edu = (w23.groupby("a28", observed=True)["цалин"]
          .agg(n="count", median="median").reindex(order))
edu["median"] /= 1e6

fig, ax = plt.subplots(figsize=(9, 6))
y = list(range(len(edu)))[::-1]                         # Доктор хамгийн дээр
bars = ax.barh(y, edu["median"], height=0.72,
               color=[color.get(k, light) for k in edu.index])
ax.set_yticks(y, [f"{k} (n={n:,})" for k, n in zip(edu.index, edu["n"])])
ax.bar_label(bars, fmt="%.2f", padding=6, fontsize=11)
ax.set_xlim(0, 3.0)                                     # 2025 оны слайдтай ижил муж
ax.set_title("Median сарын цалин, сая ₮ (n = хүний тоо)", fontsize=12)
ax.xaxis.grid(True, color="#e5e5e5"); ax.set_axisbelow(True)
ax.tick_params(axis="x", labelbottom=False, length=0)
ax.tick_params(axis="y", length=0, labelcolor="#555555")
for sp in ax.spines.values():
    sp.set_visible(False)
fig.tight_layout()
fig.savefig("figures/q3_боловсрол_2023.png", dpi=300, bbox_inches="tight", facecolor="white")

# «Гол үр дүн» хэсэгт оруулах тоонууд
b, s = edu.loc["Бакалаврын дээд", "median"], edu.loc["Бүрэн дунд", "median"]
print(f"• Бакалавр {b:.2f} сая — бүрэн дундаас ({s:.2f}) {b/s - 1:.0%} өндөр")
print(f"• Магистр {edu.loc['Магистр', 'median']:.2f} сая")
print(f"• Суурь {edu.loc['Суурь', 'median']:.2f} vs Бага {edu.loc['Бага', 'median']:.2f}")
print(f"• Доктор n={edu.loc['Доктор', 'n']} — найдвар багатай")

In [ ]:
# 2023: Насны бүлгээр цалин (2025 оны слайдтай ижил загвар)
w23["нас"] = pd.to_numeric(w23["a05"], errors="coerce")
age_labels = ["15–24", "25–34", "35–44", "45–54", "55–64", "65+"]
w23["насны_бүлэг"] = pd.cut(w23["нас"], [14, 24, 34, 44, 54, 64, 120], labels=age_labels)

age = (w23.groupby("насны_бүлэг", observed=True)["цалин"]
          .agg(n="count", median="median").reindex(age_labels))
age["median"] /= 1e6

dark, mid, light = "#134e5e", "#5f8f99", "#a0b8be"
peak = age["median"].max()
colors = [light if n < 100 else dark if m == peak else mid
          for n, m in zip(age["n"], age["median"])]

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.bar(range(len(age)), age["median"], width=0.68, color=colors)
ax.bar_label(bars, fmt="%.2f", padding=6, fontsize=12)
ax.set_xticks(range(len(age)), [f"{g}\nn={n:,}" for g, n in zip(age.index, age["n"])])
ax.set_ylim(0, 2.0)                                     # 2025 оны слайдтай ижил муж
ax.set_title("Median сарын цалин насны бүлгээр, сая ₮", fontsize=12)
ax.yaxis.grid(True, color="#e5e5e5"); ax.set_axisbelow(True)
ax.tick_params(axis="y", labelleft=False, length=0)
ax.tick_params(axis="x", length=0, labelsize=11, labelcolor="#555555")
for sp in ax.spines.values():
    sp.set_visible(False)
fig.tight_layout()
fig.savefig("figures/q3_нас_2023.png", dpi=300, bbox_inches="tight", facecolor="white")

# «Гол үр дүн» хэсэгт оруулах тоонууд
top = ", ".join(age.index[age["median"] == peak])
print(f"• 15–24: {age.loc['15–24', 'median']:.2f} сая → оргил {top}: {peak:.2f} сая")
print(f"• Дараа нь буурна: 65+ {age.loc['65+', 'median']:.2f} сая")
print(f"• 65+ n={age.loc['65+', 'n']} — найдвар багатай")

In [ ]:
# 2023: Гол үр дүн
lfpr = d["статус"].ne("Ажиллах хүчнээс гадуур").mean()
med  = w23["цалин"].median() / 1e6

g   = w23.groupby("a03", observed=True)["цалин"].median()
gap = g["Эмэгтэй"] / g["Эрэгтэй"] - 1

e   = w23.groupby("a28", observed=True)["цалин"].median()
edu = e["Бакалаврын дээд"] / e["Бүрэн дунд"] - 1

part = (d.groupby("a03", observed=True)["статус"]
          .apply(lambda s: s.ne("Ажиллах хүчнээс гадуур").mean()))

print(f"1. Ажиллах хүчний оролцоо:            {lfpr:.1%}")
print(f"2. Ердийн (median) сарын цалин:        {med:.2f} сая")
print(f"3. Эмэгтэйчүүдийн median цалин:        {gap:+.0%}")
print(f"4. Бакалавр, бүрэн дундтай харьцуулахад: {edu:+.0%}")
print(f"\nОролцоо хүйсээр: эрэгтэй {part['Эрэгтэй']:.1%}, эмэгтэй {part['Эмэгтэй']:.1%}")